[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/ananttripathi/hybrid-rag-customer-support/blob/main/Baseline_Model_Evaluation.ipynb)


# **Notebook 3: Baseline Model Evaluation**
## Assignment: Hybrid RAG & Fine-Tuning for Customer Support
---

### TO-DO: Before Running This Notebook

**Files you NEED:**
- [ ] Internet access (to download the model)
- [ ] GPU runtime enabled (Runtime → Change runtime type → T4 GPU)

**Files this notebook will CREATE:**
- [ ] `outputs.json` — `test_query`, `ground_truth`, `baseline_output` _(Required by NB4, NB5, NB7)_

---

## **Stage 3: Solution V1 (Retrieval-Assisted Generation)**

### **Task 3.1: Establish Baseline Performance**

#### **3.1.1 Execute Baseline Inference [2 marks]**
**The Task:** Load the pre-trained base model in 4-bit quantization and generate a response to an ambiguous shipping-delay query without any context.

**Hints & Tips:**
* Use `do_sample=False` for deterministic output. Do NOT pair `temperature=0.0` with `do_sample=False` — it throws a deprecation warning. Use `temperature=None, top_p=None`.
* `BitsAndBytesConfig(load_in_4bit=True)` shrinks the 1.5B model to ~750MB VRAM.
* `max_new_tokens=120` gives room for a complete answer.

**Model Selection:**
* **Qwen/Qwen2.5-1.5B-Instruct** (recommended) — must match what you used in NB2.
* **TinyLlama-1.1B-Chat** — lighter, weaker structured output.
* **Llama-3-8B-Instruct** — best quality, may OOM on free T4 during fine-tuning.

**Learner Inference:** This establishes your zero-shot baseline. Every later improvement is measured against this exact output.

In [1]:
import json
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer

MODEL_ID = "Qwen/Qwen2.5-1.5B-Instruct"  # must match Notebook 2 (Data Preparation)

DEVICE = "mps" if torch.backends.mps.is_available() else ("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {DEVICE}")

# NOTE (see Project Proposal, 1.4.1): the reference workflow loads the model via
# BitsAndBytesConfig(load_in_4bit=True) to fit a free-tier T4 GPU. bitsandbytes is CUDA-only,
# so on this Apple Silicon (MPS) execution environment we load the model in fp16 directly on
# the MPS device instead — this is a documented, deliberate hardware adaptation.
tokenizer = AutoTokenizer.from_pretrained(MODEL_ID)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

base_model = AutoModelForCausalLM.from_pretrained(MODEL_ID, dtype=torch.float16).to(DEVICE)
base_model.eval()

n_params = sum(p.numel() for p in base_model.parameters())
print(f"Loaded {MODEL_ID} in fp16 on {DEVICE}.")
print(f"Parameter count: {n_params/1e9:.2f}B")


Using device: mps


Loading weights:   0%|          | 0/338 [00:00<?, ?it/s]

Loaded Qwen/Qwen2.5-1.5B-Instruct in fp16 on mps.
Parameter count: 1.54B


In [2]:
# Deterministic inference: do_sample=False WITHOUT temperature=0.0 (that combination throws a
# deprecation warning) — instead explicitly set temperature=None, top_p=None.
GEN_KWARGS = dict(max_new_tokens=120, do_sample=False, temperature=None, top_p=None)

def generate_baseline(query, system_prompt="You are a helpful customer support assistant."):
    messages = [
        {"role": "system", "content": system_prompt},
        {"role": "user", "content": query},
    ]
    prompt = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
    inputs = tokenizer(prompt, return_tensors="pt").to(DEVICE)
    with torch.no_grad():
        out = base_model.generate(**inputs, pad_token_id=tokenizer.pad_token_id, **GEN_KWARGS)
    return tokenizer.decode(out[0][inputs["input_ids"].shape[1]:], skip_special_tokens=True).strip()

# An ambiguous, sarcastic shipping-delay query — no context provided about actual company policy.
test_query = "my package is still not here and its been forever, this is ridiculous, where even is it??"
ground_truth = "Domestic orders deliver within 3-7 business days. If a shipment exceeds this window, escalate per the Shipping Delays SOP."

baseline_output = generate_baseline(test_query)
print("TEST QUERY:", test_query)
print("\nGROUND TRUTH (SOP rule):", ground_truth)
print("\nBASELINE OUTPUT (zero-shot, no retrieval):\n", baseline_output)


TEST QUERY: my package is still not here and its been forever, this is ridiculous, where even is it??

GROUND TRUTH (SOP rule): Domestic orders deliver within 3-7 business days. If a shipment exceeds this window, escalate per the Shipping Delays SOP.

BASELINE OUTPUT (zero-shot, no retrieval):
 I'm sorry to hear that your package has not arrived yet. It's understandable if you're feeling frustrated. Here are some steps you can take:

1. Check the tracking information: Make sure you have access to the tracking number for your package. You should be able to find this on the website or app where you placed your order.

2. Contact the shipping company: If you haven't already done so, reach out to the shipping company responsible for delivering your package. They may be able to provide more information about why your package hasn't arrived yet.

3. Check your address: Double-check


#### **3.1.2 Evaluate Baseline Quality [2 marks]**
**The Task:** Assess the baseline output for factual inaccuracies against the ground-truth SOP rule.

**Hints & Tips:**
* Compare against the known rule: "Domestic orders deliver within 3-7 business days."
* Did the model invent a timeline? Mention a non-existent tracking system or department?
* Document every hallucination — it justifies Stages 3 and 4.

**Learner Inference:** This hallucination is exactly why you build Stage 3 (a database) and Stage 4 (a router).

In [3]:
print("Ground-truth rule: Domestic orders deliver within 3-7 business days (see shipping_delays.md).")
print("\nBaseline output:\n", baseline_output)

hallucination_flags = []
lower_out = baseline_output.lower()

if any(kw in lower_out for kw in ["24 hour", "24-hour", "1-2 day", "next day", "overnight", "within 48"]):
    hallucination_flags.append("Invents a delivery timeline that does not match the 3-7 business day SOP.")
if any(kw in lower_out for kw in ["tracking number", "tracking id", "click here", "track your order at", "app"]):
    hallucination_flags.append("References a tracking mechanism/URL/app not described in any provided SOP.")
if "department" in lower_out or "team" in lower_out:
    hallucination_flags.append("Mentions a specific department/team not defined in the escalation_matrix SOP.")
if not hallucination_flags:
    hallucination_flags.append(
        "No obviously fabricated specifics in this single sample, but the response also never cites "
        "the actual '3-7 business day' rule — it is functionally correct by accident, not by design. "
        "See Notebook 5 for a systematic hallucination-frequency measurement across the full test set."
    )

print("\nHallucination / quality assessment:")
for flag in hallucination_flags:
    print(" -", flag)

did_cite_correct_window = "3" in baseline_output and "7" in baseline_output
print(f"\nDid the baseline correctly cite the '3-7 business day' window? {did_cite_correct_window}")
print(
    "\nConclusion: without any grounding in company policy, the baseline model must guess at delivery "
    "windows, tracking mechanisms, and escalation paths it has never actually seen. This motivates "
    "Stage 3 (retrieval-assisted generation) and Stage 4 (fine-tuned intent routing)."
)


Ground-truth rule: Domestic orders deliver within 3-7 business days (see shipping_delays.md).

Baseline output:
 I'm sorry to hear that your package has not arrived yet. It's understandable if you're feeling frustrated. Here are some steps you can take:

1. Check the tracking information: Make sure you have access to the tracking number for your package. You should be able to find this on the website or app where you placed your order.

2. Contact the shipping company: If you haven't already done so, reach out to the shipping company responsible for delivering your package. They may be able to provide more information about why your package hasn't arrived yet.

3. Check your address: Double-check

Hallucination / quality assessment:
 - References a tracking mechanism/URL/app not described in any provided SOP.

Did the baseline correctly cite the '3-7 business day' window? False

Conclusion: without any grounding in company policy, the baseline model must guess at delivery windows, trac

---
## Save Artifacts for Downstream Notebooks

**IMPORTANT:** Saves the baseline output. Notebooks 4, 5, and 7 depend on this file.

In [4]:
outputs = {
    "test_query": test_query,
    "ground_truth": ground_truth,
    "baseline_output": baseline_output,
}
with open("outputs.json", "w") as f:
    json.dump(outputs, f, indent=2)

print("Saved outputs.json:")
print(json.dumps(outputs, indent=2))


Saved outputs.json:
{
  "test_query": "my package is still not here and its been forever, this is ridiculous, where even is it??",
  "ground_truth": "Domestic orders deliver within 3-7 business days. If a shipment exceeds this window, escalate per the Shipping Delays SOP.",
  "baseline_output": "I'm sorry to hear that your package has not arrived yet. It's understandable if you're feeling frustrated. Here are some steps you can take:\n\n1. Check the tracking information: Make sure you have access to the tracking number for your package. You should be able to find this on the website or app where you placed your order.\n\n2. Contact the shipping company: If you haven't already done so, reach out to the shipping company responsible for delivering your package. They may be able to provide more information about why your package hasn't arrived yet.\n\n3. Check your address: Double-check"
}


---
## END-OF-NOTEBOOK CHECKLIST

> **IMPORTANT: Verify before proceeding to Notebook 4.**

- [ ] Base model loaded in 4-bit without errors
- [ ] Baseline output generated for `test_query`
- [ ] Hallucination assessment documented
- [ ] **`outputs.json` saved** with `test_query`, `ground_truth`, `baseline_output` ← _CRITICAL for NB4, 5, 7_
- [ ] GPU runtime enabled

**If any item is unchecked, fix it before moving on.**